In [1]:
from pathlib import Path
import pandas as pd

# Proje kökünü bul
cwd = Path.cwd()
ROOT = cwd if (cwd / "notebooks").exists() else cwd.parent

DATA_DIR = ROOT / "data"

print("Project root:", ROOT)
print("\nAvailable data files:")

for file in sorted(DATA_DIR.iterdir()):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"{file.name} - {size_mb:.1f} MB")

Project root: c:\anaconda13\Scripts\PAYSIM

Available data files:
accounts.csv - 4.1 MB
model_validation_comparison.csv - 0.0 MB
paysim.csv - 470.7 MB
paysim_canonical_200k.csv - 17.2 MB
paysim_download.zip - 177.8 MB
paysim_model_200k.csv - 15.4 MB
sent_relationships.csv - 4.2 MB
to_relationships.csv - 4.2 MB
transaction_features_graph_v1.csv - 23.0 MB
transaction_features_temporal_v1.csv - 24.8 MB
transaction_features_v1.csv - 20.5 MB
transactions.csv - 17.2 MB


In [2]:
# Orijinal PaySim verisinden yalnızca gerekli kolonları oku
df = pd.read_csv(
    DATA_DIR / "paysim.csv",
    usecols=["step", "type", "isFraud"],
    dtype={
        "step": "int16",
        "type": "category",
        "isFraud": "int8"
    }
)

# V1 ile aynı transaction scope
scoped_df = df.loc[
    df["type"].isin(["CASH_OUT", "TRANSFER"])
].copy()

print("Original rows:", len(df))
print("Scoped rows:", len(scoped_df))
print("First time step:", scoped_df["step"].min())
print("Last time step:", scoped_df["step"].max())
print("Unique time steps:", scoped_df["step"].nunique())

Original rows: 6362620
Scoped rows: 2770409
First time step: 1
Last time step: 743
Unique time steps: 743


In [3]:
# Etiketlere bakmadan temporal split sınırlarını belirle

step_counts = (
    scoped_df.groupby("step")
    .size()
    .sort_index()
)

cumulative = step_counts.cumsum()
total = len(scoped_df)

# Hedef oranlar
train_target = total * 0.70
train_val_target = total * 0.85

# Hedef satır sayılarına en yakın time_step sınırları
train_end_step = (cumulative - train_target).abs().idxmin()
val_end_step = (cumulative - train_val_target).abs().idxmin()

assert train_end_step < val_end_step

train_count = cumulative.loc[train_end_step]
val_count = cumulative.loc[val_end_step] - train_count
test_count = total - cumulative.loc[val_end_step]

print("Train ends at step:", train_end_step)
print("Validation ends at step:", val_end_step)

print(f"\nTrain: {train_count:,} ({train_count/total:.2%})")
print(f"Validation: {val_count:,} ({val_count/total:.2%})")
print(f"Test: {test_count:,} ({test_count/total:.2%})")

Train ends at step: 322
Validation ends at step: 376

Train: 1,938,484 (69.97%)
Validation: 415,628 (15.00%)
Test: 416,297 (15.03%)


In [4]:
# Frozen temporal split diagnostics

partitions = {
    "Train": scoped_df["step"] <= train_end_step,
    "Validation": scoped_df["step"].between(
        train_end_step + 1, val_end_step
    ),
    "Test": scoped_df["step"] > val_end_step,
}

results = []

for name, mask in partitions.items():
    labels = scoped_df.loc[mask, "isFraud"]

    results.append({
        "partition": name,
        "transactions": len(labels),
        "fraud": int(labels.sum()),
        "fraud_rate_pct": round(labels.mean() * 100, 4),
    })

diagnostics = pd.DataFrame(results)

display(diagnostics)

,partition,transactions,fraud,fraud_rate_pct
0,Train,1938484,3633,0.1874
1,Validation,415628,560,0.1347
2,Test,416297,4020,0.9657


In [2]:
from pathlib import Path

# Çalışma konumumuza göre proje kökünü belirledik.
cwd = Path.cwd()
ROOT = cwd if (cwd / "data").exists() else cwd.parent

# Veri klasörünün yolunu oluşturduk.
DATA_DIR = ROOT / "data"

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)

Project root: c:\anaconda13\Scripts\PAYSIM
Data directory: c:\anaconda13\Scripts\PAYSIM\data


In [3]:
import sys
import pandas as pd

# Proje kökünü Python'ın arama yoluna ekledik.
sys.path.insert(0, str(ROOT))

# Hazırladığımız PaySim dönüşüm fonksiyonunu içe aktardık.
from src.data.paysim import prepare_scoped_paysim

# Hızlı bir kontrol için orijinal verinin ilk 100 satırını kullandık.
sample = pd.read_csv(DATA_DIR / "paysim.csv", nrows=100)

# Veriyi canonical şemamıza dönüştürdük.
prepared = prepare_scoped_paysim(sample)

print("Original rows:", len(sample))
print("Prepared rows:", len(prepared))
print("Columns:", prepared.columns.tolist())

display(prepared.head())

Original rows: 100
Prepared rows: 34
Columns: ['tx_id', 'time_step', 'tx_type', 'amount', 'src_id', 'src_balance_before', 'src_balance_after', 'dst_id', 'dst_balance_before', 'dst_balance_after', 'label_fraud', 'label_flagged']


,tx_id,time_step,tx_type,amount,src_id,src_balance_before,src_balance_after,dst_id,dst_balance_before,dst_balance_after,label_fraud,label_flagged
0,raw_3,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0
1,raw_4,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0
2,raw_16,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0
3,raw_20,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0
4,raw_25,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0


In [4]:
# Gerekli kolonların oluşturulduğunu kontrol ettik.
assert "tx_id" in prepared.columns
assert "src_id" in prepared.columns
assert "dst_id" in prepared.columns
assert "time_step" in prepared.columns
assert "label_fraud" in prepared.columns

# İşlem türlerini ve transaction ID benzersizliğini doğruladık.
assert prepared["tx_type"].isin(["CASH_OUT", "TRANSFER"]).all()
assert prepared["tx_id"].is_unique

print("All basic checks passed!")

All basic checks passed!
